# Anomaly Detection & Risk Identification

This notebook implements threshold-based monitoring, statistical Z-score detection, severity classification, anomaly logging, and visualization using the available HirePulse transaction data.

## 1. Load Transaction Data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data' / 'raw' / 'transactions.csv'
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df = df.dropna(subset=['transaction_date', 'amount']).copy()

print(f'Transactions: {len(df)}')
print(f'Date range: {df["transaction_date"].min()} to {df["transaction_date"].max()}')
df

## 2. Build Daily Revenue Metrics

Daily revenue is calculated from the actual transaction amounts.

In [ ]:
daily = (
    df.assign(date=df['transaction_date'].dt.normalize())
    .groupby('date')
    .agg(
        daily_revenue=('amount', 'sum'),
        transaction_count=('customer_id', 'count')
    )
    .sort_index()
)

daily

## 3. 30-Day Lookback Window

The assignment requests a 30-day lookback. The repository contains fewer than 30 days of transaction history, so all available observations within the requested window are used.

In [ ]:
latest_date = daily.index.max()
lookback_start = latest_date - pd.Timedelta(days=29)

daily_30 = daily[daily.index >= lookback_start].copy()

print(f'Lookback: {lookback_start.date()} to {latest_date.date()}')
print(f'Available observations: {len(daily_30)}')
display(daily_30)

## Task 1 — Threshold-Based Anomaly Detection

Business thresholds are applied to daily revenue and transaction count. Signup rate is not available in the transaction dataset, so it is not fabricated.

In [ ]:
alert_rules = {
    'daily_revenue': {'min': 5000, 'max': 50000},
    'transaction_count': {'min': 100, 'max': 10000},
    'signup_rate': {'min': 10, 'max': 500}
}

def check_thresholds(metrics, rules):
    alerts = []
    for metric_name, rule in rules.items():
        if metric_name not in metrics:
            continue
        value = metrics[metric_name]
        if value < rule['min']:
            alerts.append({
                'metric': metric_name,
                'value': value,
                'threshold': rule['min'],
                'direction': 'BELOW_MIN',
                'severity': 'HIGH'
            })
        elif value > rule['max']:
            alerts.append({
                'metric': metric_name,
                'value': value,
                'threshold': rule['max'],
                'direction': 'ABOVE_MAX',
                'severity': 'MEDIUM'
            })
    return alerts

current_metrics = {
    'daily_revenue': daily_30['daily_revenue'].iloc[-1],
    'transaction_count': daily_30['transaction_count'].iloc[-1]
}

threshold_alerts = check_thresholds(current_metrics, alert_rules)
threshold_alerts

### Threshold Result

The latest available day triggers alerts because daily revenue and transaction count are below the assignment's minimum thresholds.

## Task 2 — Statistical Z-Score Detection

Values beyond ±2 standard deviations from the mean are flagged as statistical anomalies.

In [ ]:
daily_revenue = daily_30['daily_revenue']
mean = daily_revenue.mean()
std = daily_revenue.std()

if pd.isna(std) or std == 0:
    z_scores = pd.Series(0.0, index=daily_revenue.index)
    anomalies = daily_revenue.iloc[0:0]
else:
    z_scores = (daily_revenue - mean) / std
    anomalies = daily_revenue[z_scores.abs() > 2]

print(f'Mean revenue: ${mean:.2f}')
print(f'Standard deviation: ${std:.2f}')
print(f'Anomalies detected: {len(anomalies)}')

for date, value in anomalies.items():
    print(f'{date.date()}: ${value:.2f}, z-score={z_scores[date]:.2f}')

## Task 3 — Severity Classification

Severity is based on z-score magnitude:

- CRITICAL: > 3σ
- HIGH: > 2σ
- MEDIUM: > 1.5σ
- LOW: otherwise

In [ ]:
def classify_severity(value, mean, std):
    if pd.isna(std) or std == 0:
        return 'LOW'
    z = abs((value - mean) / std)
    if z > 3:
        return 'CRITICAL'
    elif z > 2:
        return 'HIGH'
    elif z > 1.5:
        return 'MEDIUM'
    return 'LOW'

severity_records = []
for date, value in anomalies.items():
    severity_records.append({
        'date': date,
        'value': value,
        'z_score': z_scores[date],
        'severity': classify_severity(value, mean, std)
    })

severity_df = pd.DataFrame(severity_records)
severity_df

In [ ]:
if not severity_df.empty:
    high_severity = severity_df[
        severity_df['severity'].isin(['CRITICAL', 'HIGH'])
    ]
else:
    high_severity = severity_df

print(f'High-severity anomalies requiring investigation: {len(high_severity)}')

## Task 4 — Anomaly Logging and Audit Trail

In [ ]:
anomaly_log = []

for date, value in anomalies.items():
    severity = classify_severity(value, mean, std)
    anomaly_log.append({
        'timestamp': pd.Timestamp.now(),
        'anomaly_date': date,
        'metric': 'daily_revenue',
        'value': value,
        'expected_range': f'{mean - 2 * std:.2f}-{mean + 2 * std:.2f}',
        'z_score': z_scores[date],
        'severity': severity,
        'status': 'OPEN'
    })

anomalies_df = pd.DataFrame(anomaly_log, columns=[
    'timestamp', 'anomaly_date', 'metric', 'value',
    'expected_range', 'z_score', 'severity', 'status'
])

ANOMALY_LOG_PATH = OUTPUT_DIR / 'anomalies_log.csv'
anomalies_df.to_csv(ANOMALY_LOG_PATH, index=False)

print(f'Logged {len(anomalies_df)} anomalies')
print(f'Saved to: {ANOMALY_LOG_PATH}')
display(anomalies_df)

## Task 5 — Visualization

The chart shows raw daily revenue, a 7-day moving average, the expected ±2σ range, and any detected anomalies.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    daily_30.index,
    daily_30['daily_revenue'],
    marker='o',
    linewidth=2,
    label='Daily Revenue'
)

rolling_avg = daily_30['daily_revenue'].rolling(
    window=7,
    min_periods=1
).mean()

ax.plot(
    rolling_avg.index,
    rolling_avg.values,
    linewidth=2,
    label='7-day Moving Average'
)

if not pd.isna(std) and std > 0:
    ax.fill_between(
        daily_30.index,
        mean - 2 * std,
        mean + 2 * std,
        alpha=0.2,
        label='Expected Range ±2σ'
    )

if len(anomalies) > 0:
    ax.scatter(
        anomalies.index,
        anomalies.values,
        s=200,
        marker='X',
        zorder=5,
        label='Anomaly'
    )

    for date, value in anomalies.items():
        ax.annotate(
            'ANOMALY',
            (date, value),
            xytext=(0, 10),
            textcoords='offset points',
            ha='center',
            fontweight='bold'
        )

ax.set_xlabel('Date')
ax.set_ylabel('Revenue ($)')
ax.set_title('Daily Revenue with Anomalies Flagged')
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

PLOT_PATH = OUTPUT_DIR / 'anomaly_detection.png'
plt.savefig(PLOT_PATH, dpi=150)
plt.show()

## Monitoring Interpretation

The threshold method and statistical method answer different questions. Threshold rules compare metrics with business-defined limits, while Z-score detection identifies observations that are statistically unusual relative to the available history.

For this repository, the latest daily revenue and transaction count are below the business thresholds. However, none of the four available daily revenue observations exceeds ±2 standard deviations, so the statistical detector reports zero anomalies.

Because the repository contains only four transaction observations across fewer than 30 days, the statistical result should not be interpreted as proof that the system is operating normally. More historical data is required for reliable monitoring.